
# Checkpointed multi-step gradient calibration (prototype)

Uses `compute_gradients_checkpointed!` (`src/gradients_checkpointed.jl`) instead of the
single-timestep `compute_gradients!` that `calibrate!` is built on.

**The simplification this notebook makes**: `calibrate!`'s batch loop exists to *approximate*
a longer-horizon gradient by averaging many single-timestep gradient samples
(`samples_per_batch`) taken across a window of undifferentiated real time (`batch_days`).
A checkpointed multi-step gradient doesn't need that approximation — one
`compute_gradients_checkpointed!` call *is* a real gradient spanning `N` timesteps, so all of
`batch_days` / `samples_per_batch` / `steps_per_sample` / gradient-sample-averaging falls
away. One training iteration = one checkpointed `N`-step gradient, full stop. Convergence
detection (smoothed-loss threshold, best-point tracking, plateau LR decay) is kept — that's
not batching machinery, it's how `calibrate!` knows when to stop, and this notebook needs the
same thing.

This intentionally does **not** reuse `calibrate!`: it's a standalone loop over the low-level
pieces (`ParamSpec`, `LossConfig`, `compute_gradients_checkpointed!`, the sigmoid
reparameterisation helpers), so the structure stays visible and easy to change while this is
still a prototype.

**Before running**: read `examples/checkpointed_multistep_gradients/TODO.md`. Two things from
there matter directly here:
- The **first** call to `compute_gradients_checkpointed!` triggers Enzyme's compilation of the
  whole checkpointed loop. Measured at ~3 hours for this model config (T31/L8, 15 params,
  N=5) — this is a one-time, per-Julia-session cost; every call after the first (same `N` and
  parameter set) is seconds. Don't panic at the first iteration's wall-clock time, and don't
  restart the kernel between iterations or you'll pay it again.
- The gradient itself is known to blow up to all-NaN well before N≈200 on the raw one-hot
  example (not yet re-checked with this real flux-loss seed). `N` here defaults small
  (`N = 5`) for that reason — this is deliberately not yet an attempt at the "20-day gradient"
  the `lrd`/`srd` investigation actually wants, just a working scaffold to build that towards
  once the NaN ceiling is characterized for the real loss.


In [ ]:
using Pkg
Pkg.activate(joinpath(@__DIR__, "..", ".."))

using SpeedyCalibration, SpeedyWeather, Optimisers, Dates, Printf, Statistics, CairoMakie


## Model, loss, and trainable parameters

The **same 15 parameters, loss targets, and resolution** as
`thesis_15param_shortwave.ipynb` (the canonical `calibrate!`-based reproduction of the
thesis's Chapter 5 multi-parameter shortwave calibration) — cloud reflection (3),
cloud cover (1), atmospheric absorption (4), land albedo (5), ocean/ice albedo (2).
Using the same set here isn't optional padding: compile cost is driven by
differentiating the whole `model`, not by how many `ParamSpec`s are read out of it
afterward (`get_by_path(dmodel, spec.path)` is a cheap lookup into an already-computed
gradient), so there's no compile-cost reason to use fewer.

**What's genuinely different from the canonical run**: the optimizer learning rate.
The canonical run uses `Adam(1f-1)` tuned for gradients that are an *average of many
single-timestep samples* (`samples_per_batch=15` per batch) — a very different noise/
magnitude regime than one checkpointed `N`-step gradient per iteration. Blindly reusing
`1f-1` here would be copying a hyperparameter tuned for a different gradient estimator;
it's left smaller and flagged as needing real tuning once N-step gradients are known to
behave reasonably (see the NaN-blowup caveat in `TODO.md`).


In [ ]:
param_specs = [
    # ── Cloud reflection (3) ──────────────────────────────────────────────────
    ParamSpec(:cloud_albedo,
        [:shortwave_radiation, :clouds, :cloud_albedo];
        bounds=(0.25f0, 0.95f0), initial=0.60f0),

    ParamSpec(:stratocumulus_cover_max,
        [:shortwave_radiation, :clouds, :stratocumulus_cover_max];
        bounds=(0.25f0, 0.95f0), initial=0.60f0),

    ParamSpec(:stratocumulus_albedo,
        [:shortwave_radiation, :clouds, :stratocumulus_albedo];
        bounds=(0.10f0, 0.90f0), initial=0.50f0),

    # ── Cloud cover (1) ───────────────────────────────────────────────────────
    ParamSpec(:precipitation_weight,
        [:shortwave_radiation, :clouds, :precipitation_weight];
        bounds=(0.0f0, 0.8f0), initial=0.20f0),

    # ── Atmospheric absorption (4) ───────────────────────────────────────────
    # absorptivity_water_vapor has a ~200x weaker physical gradient than the
    # cloud parameters (it is multiplied by specific humidity q ~ 0.005 in the
    # radiation scheme). Lower bound raised to 60 (from 10): the model becomes
    # numerically unstable below ~57.
    ParamSpec(:absorptivity_water_vapor,
        [:shortwave_radiation, :transmissivity, :absorptivity_water_vapor];
        bounds=(60f0, 140f0), initial=75f0),

    ParamSpec(:absorptivity_dry_air,
        [:shortwave_radiation, :transmissivity, :absorptivity_dry_air];
        bounds=(0.005f0, 0.060f0), initial=0.03135f0),

    ParamSpec(:absorptivity_aerosol,
        [:shortwave_radiation, :transmissivity, :absorptivity_aerosol];
        bounds=(0.005f0, 0.060f0), initial=0.03135f0),

    ParamSpec(:ozone_absorption,
        [:shortwave_radiation, :radiative_transfer, :ozone_absorption];
        bounds=(0.002f0, 0.020f0), initial=0.01f0),

    # ── Land surface albedo (5) ──────────────────────────────────────────────
    ParamSpec(:albedo_land,
        [:albedo, :land, :albedo_land];
        bounds=(0.10f0, 0.70f0), initial=0.40f0),

    ParamSpec(:albedo_high_vegetation,
        [:albedo, :land, :albedo_high_vegetation];
        bounds=(0.04f0, 0.26f0), initial=0.15f0),

    ParamSpec(:albedo_low_vegetation,
        [:albedo, :land, :albedo_low_vegetation];
        bounds=(0.05f0, 0.35f0), initial=0.20f0),

    ParamSpec(:albedo_snow,
        [:albedo, :land, :albedo_snow];
        bounds=(0.15f0, 0.75f0), initial=0.40f0),

    ParamSpec(:snow_depth_scale,
        [:albedo, :land, :snow_depth_scale];
        bounds=(0.005f0, 0.20f0), initial=0.05f0),

    # ── Ocean/ice surface albedo (2) ─────────────────────────────────────────
    ParamSpec(:albedo_ocean,
        [:albedo, :ocean, :albedo_ocean];
        bounds=(0.02f0, 0.10f0), initial=0.06f0),

    ParamSpec(:albedo_ice,
        [:albedo, :ocean, :albedo_ice];
        bounds=(0.30f0, 0.90f0), initial=0.60f0),
]

# Targets from Wild et al. (2015) / Trenberth (2009) -- same as thesis_15param_shortwave.ipynb
loss_config = LossConfig([:osr, :sru, :srd];
    targets = Dict(:osr => 101.9f0, :sru => 23.1f0, :srd => 184.3f0),
    weights = Dict(:osr => 1f0,     :sru => 1f0,     :srd => 1f0))

n_params = length(param_specs)
param_names = [spec.name for spec in param_specs]
println("$(n_params) parameters, loss: OSR=$(loss_config.targets[:osr]) SRU=$(loss_config.targets[:sru]) SRD=$(loss_config.targets[:srd]) W/m² (equal weights)")


## Build the model and spin up

`daily_cycle = true` — standing rule for this project, not a config choice (see main
`MEMORY.md`).


In [ ]:
sg     = SpectralGrid(trunc=31, nlayers=8)
planet = Earth(sg; daily_cycle=true, seasonal_cycle=false)
model  = PrimitiveWetModel(sg; planet=planet)

# apply the ParamSpecs' initial values before spinup, same convention as calibrate!
p = vec(parameters(model))
init_phys = Float32[]
for spec in param_specs
    val = isnothing(spec.initial) ? Float32(get_by_path(p, spec.path)) : spec.initial
    set_by_path!(p, spec.path, val)
    push!(init_phys, val)
end
model = SpeedyWeather.reconstruct(model, p)

sim = initialize!(model)
sim.variables.prognostic.clock.time = DateTime(2000, 3, 21)   # same as thesis_15param_shortwave.ipynb
SpeedyWeather.initialize!(sim; period=Day(365*100), output=false)

spinup_days = 20   # same as thesis_15param_shortwave.ipynb
clock = sim.variables.prognostic.clock
steps_per_day = ceil(Int, Millisecond(Day(1)).value / Millisecond(clock.Δt).value)

println("Spinning up $spinup_days days...")
t0 = time()
for _ in 1:(spinup_days * steps_per_day)
    SpeedyWeather.time_step!(sim)
end
@printf("Spinup complete in %.1f s.\n", time() - t0)


## Optimizer setup (sigmoid reparameterisation, same as `calibrate!`)

Bounds are enforced structurally: the optimizer moves in unconstrained `θ_raw`, mapped to the
physical value via a sigmoid, so step sizes shrink naturally near the bounds instead of being
clamped (which would break Adam's momentum state).


In [ ]:
opt_params  = Float32[to_raw(init_phys[i], spec.bounds[1], spec.bounds[2])
                      for (i, spec) in enumerate(param_specs)]
phys_values = Float32[sigmoid_param(opt_params[i], spec.bounds[1], spec.bounds[2])
                      for (i, spec) in enumerate(param_specs)]

N           = 5       # checkpointed steps per iteration -- see TODO.md before raising this
init_lr     = 5f-3
opt_state   = Optimisers.setup(Optimisers.Adam(init_lr), opt_params)
current_lr  = init_lr
grad_clip   = 1f2     # safety net given the documented blow-up risk, not fine-tuned

# Convergence: same convention/defaults as calibrate!'s TrainingConfig (loss_window_size=20,
# loss_threshold=1f0 -- directly comparable since this uses the same loss_config as
# thesis_15param_shortwave.ipynb), just driven by iterations of the checkpointed gradient
# instead of batches of averaged single-step gradients.
max_iterations      = 1000    # safety cap, not a target -- training stops on convergence, not this
loss_window_size    = 20
loss_threshold      = 1f0
enable_lr_decay     = true
lr_decay_factor     = 0.5f0
lr_plateau_patience = 50
min_lr              = 1f-6
max_lr_decays       = 3

loss_window         = Float32[]
best_smoothed_loss  = Inf32
best_phys_values    = copy(phys_values)
best_iteration      = 0
iters_since_best    = 0
lr_decay_count      = 0
converged           = false
stop_reason         = "max_iterations reached"

history = Dict{Symbol,Vector{Float32}}(
    :iteration => Float32[], :loss => Float32[], :smoothed_loss => Float32[],
    :elapsed_time => Float32[], :lr => Float32[],
)
for k in loss_config.flux_keys
    history[k] = Float32[]
end
for name in param_names
    history[name]                     = Float32[]
    history[Symbol("grad_", name)]    = Float32[]
end


## Training loop

Runs until convergence (smoothed loss below `loss_threshold`), same criterion `calibrate!`
uses, up to `max_iterations` as a safety cap rather than a target. One iteration = one
checkpointed `N`-step gradient (`compute_gradients_checkpointed!` internally restores
`sim.variables`/`sim.model` to where it found them — it's a pure query), one Adam step in
sigmoid space, then `model` is rebuilt via `SpeedyWeather.reconstruct` with the new physical
parameter values (mutating `model`'s parameters isn't how SpeedyWeather works; `calibrate!`
does the same rebuild-and-swap), and finally the simulation is advanced `N` real
(undifferentiated) steps so the *next* iteration's gradient starts from a new point in time —
this is what "one gradient over a batch" means here: `N` steps of real elapsed time per
iteration, differentiated exactly once, no sample-averaging.

Non-finite loss/gradients (the documented NaN-blowup risk) are caught and skipped rather than
corrupting the optimizer state — the simulation still advances so training can continue past a
bad iteration, but no parameter update happens on one. Plateau-triggered LR decay is included
(same as `calibrate!`) since that's a convergence aid, not batching machinery.


In [ ]:
start_time = time()

for iter in 1:max_iterations
    grads, means, loss = compute_gradients_checkpointed!(
        sim.variables, sim.model, loss_config, param_specs, N)

    if !isfinite(loss) || any(!isfinite, grads)
        @printf("Iteration %4d: non-finite loss/gradient (loss=%.3g) -- skipping update.\n",
                iter, loss)
        # keep all history vectors aligned by iteration: log the *unchanged* phys_values and
        # NaN gradients rather than omitting this iteration from the per-param arrays.
        for (i, name) in enumerate(param_names)
            push!(history[name],                  phys_values[i])
            push!(history[Symbol("grad_", name)], NaN32)
        end
        # advance anyway so training can continue past a bad iteration
        for _ in 1:N; SpeedyWeather.time_step!(sim); end
        push!(history[:iteration],     Float32(iter))
        push!(history[:loss],          loss)
        push!(history[:smoothed_loss], isempty(loss_window) ? loss : mean(loss_window))
        push!(history[:elapsed_time],  Float32(time() - start_time))
        push!(history[:lr],            current_lr)
        for k in loss_config.flux_keys; push!(history[k], means[k]); end
        continue
    end

    # per-param grad_scale and sigmoid chain-rule factor, same convention as calibrate!
    scaled_grads = Float32[
        grads[i] * param_specs[i].grad_scale *
        sigmoid_grad_factor(opt_params[i], param_specs[i].bounds[1], param_specs[i].bounds[2])
        for i in 1:n_params
    ]

    grad_norm = sqrt(sum(scaled_grads .^ 2))
    if grad_norm > grad_clip
        scaled_grads .*= grad_clip / grad_norm
    end

    global opt_state, opt_params, sim, phys_values, current_lr
    global best_smoothed_loss, best_phys_values, best_iteration, iters_since_best, lr_decay_count
    global converged, stop_reason

    opt_state, opt_params = Optimisers.update(opt_state, opt_params, scaled_grads)
    phys_values = Float32[sigmoid_param(opt_params[i], spec.bounds[1], spec.bounds[2])
                          for (i, spec) in enumerate(param_specs)]

    # rebuild model with the updated parameters (SpeedyWeather.reconstruct, same as calibrate!)
    new_p = vec(parameters(sim.model))
    for (i, spec) in enumerate(param_specs)
        set_by_path!(new_p, spec.path, phys_values[i])
    end
    # `Leapfrog.first_step_euler` was removed in SpeedyWeather 0.22 -- `prognostic_step`
    # now decides via `clock.step_counter <= 1` instead, and `reconstruct` doesn't touch
    # the clock, so no equivalent reset is needed here any more.
    updated_model = SpeedyWeather.reconstruct(sim.model, new_p)
    sim = Simulation(sim.variables, updated_model)

    for (i, name) in enumerate(param_names)
        push!(history[name],                  phys_values[i])
        push!(history[Symbol("grad_", name)], grads[i])
    end

    # smoothed loss + plateau detection, same convention as calibrate!
    push!(loss_window, loss)
    length(loss_window) > loss_window_size && popfirst!(loss_window)
    smoothed_loss = mean(loss_window)

    if smoothed_loss < best_smoothed_loss
        best_smoothed_loss = smoothed_loss
        best_phys_values   = copy(phys_values)
        best_iteration     = iter
        iters_since_best   = 0
    else
        iters_since_best  += 1
    end

    if enable_lr_decay && iters_since_best >= lr_plateau_patience &&
            current_lr > min_lr && lr_decay_count < max_lr_decays
        old_lr = current_lr
        current_lr = max(current_lr * lr_decay_factor, min_lr)
        opt_state = Optimisers.setup(Optimisers.Adam(current_lr), opt_params)
        iters_since_best = 0
        lr_decay_count += 1
        @printf("  ↓ LR: %.2e → %.2e (decay #%d)\n", old_lr, current_lr, lr_decay_count)
    end

    push!(history[:iteration],     Float32(iter))
    push!(history[:loss],          loss)
    push!(history[:smoothed_loss], smoothed_loss)
    push!(history[:elapsed_time],  Float32(time() - start_time))
    push!(history[:lr],            current_lr)
    for k in loss_config.flux_keys
        push!(history[k], means[k])
    end

    if iter <= 10 || iter % 5 == 0
        flux_str = join([@sprintf("%s=%6.2f", k, means[k]) for k in loss_config.flux_keys], " ")
        @printf("Iteration %4d | LR %.1e | %s | L̄ %8.3f | elapsed=%.1fs\n",
                iter, current_lr, flux_str, smoothed_loss, time() - start_time)
    end

    # advance N real (undifferentiated) steps so the next iteration starts from a new point
    for _ in 1:N
        SpeedyWeather.time_step!(sim)
    end

    if smoothed_loss < loss_threshold
        converged   = true
        stop_reason = "smoothed loss below threshold ($loss_threshold)"
        @printf("CONVERGED at iteration %d: smoothed_loss %.4f < %.4f\n",
                iter, smoothed_loss, loss_threshold)
        break
    end
end

println(stop_reason, converged ? "" : " (did not converge)")
@printf("Best smoothed loss %.4f at iteration %d\n", best_smoothed_loss, best_iteration)


## Plots


In [ ]:
fig = Figure(size=(1000, 700))

ax_loss = Axis(fig[1, 1:2]; xlabel="iteration", ylabel="loss", title="Loss")
lines!(ax_loss, history[:iteration], history[:loss]; color=(:steelblue, 0.4), label="raw")
lines!(ax_loss, history[:iteration], history[:smoothed_loss]; color=:steelblue, label="smoothed")
hlines!(ax_loss, [loss_threshold]; linestyle=:dash, color=:red, label="threshold")
axislegend(ax_loss)

ax_flux = Axis(fig[2, 1:2]; xlabel="iteration", ylabel="W/m²", title="Flux means")
for k in loss_config.flux_keys
    lines!(ax_flux, history[:iteration], history[k]; label=String(k))
    hlines!(ax_flux, [loss_config.targets[k]]; linestyle=:dash)
end
axislegend(ax_flux)

for (i, name) in enumerate(param_names)
    ax = Axis(fig[3 + (i-1) ÷ 4, 1 + (i-1) % 4]; xlabel="iteration", title=String(name))
    lines!(ax, history[:iteration], history[name])
end

fig
